# Data Sorting and Loading
Load and process both JSON and Parquet files from the repository.

In [ ]:
import json
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
import statsmodels.api as sm
from sklearn.preprocessing import LabelEncoder

## Loading JSON Files

In [ ]:
# List of JSON files
json_files = ['matches.json', 'match_ids.json', 'momentum.json']

# Load each JSON file
json_data = {}
for json_file in json_files:
    if os.path.exists(json_file):
        with open(json_file, 'r') as f:
            json_data[json_file] = json.load(f)
        print(f'Loaded {json_file}: {type(json_data[json_file])} with {len(json_data[json_file])} items')
    else:
        print(f'File not found: {json_file}')

## Loading Parquet Files

In [ ]:
# List of Parquet files
parquet_files = ['acclimatization.parquet', 'historical_placebo.parquet', 'stoppages.parquet']

# Load each Parquet file as a DataFrame
parquet_data = {}
for parquet_file in parquet_files:
    if os.path.exists(parquet_file):
        parquet_data[parquet_file] = pd.read_parquet(parquet_file)
        print(f'Loaded {parquet_file}: DataFrame with shape {parquet_data[parquet_file].shape}')
        print(f'Columns: {list(parquet_data[parquet_file].columns)}')
    else:
        print(f'File not found: {parquet_file}')

## Merge Matches and Momentum Data

In [ ]:
# Convert JSON data to DataFrames for easier merging
matches_df = pd.DataFrame(json_data['matches.json'])
momentum_df = pd.DataFrame(json_data['momentum.json'])
match_ids = json_data['match_ids.json']

print(f'Matches DataFrame shape: {matches_df.shape}')
print(f'Momentum DataFrame shape: {momentum_df.shape}')
print(f'Number of match IDs: {len(match_ids)}')
print(f'\nMatches columns: {list(matches_df.columns)}')
print(f'Momentum columns: {list(momentum_df.columns)}')

In [ ]:
# Convert match_ids to strings for consistent comparison
match_ids_str = [str(mid) for mid in match_ids]

# Filter matches and momentum to only include those in match_ids
matches_filtered = matches_df[matches_df['id'].astype(str).isin(match_ids_str)].copy()
momentum_filtered = momentum_df[momentum_df['id'].astype(str).isin(match_ids_str)].copy()

print(f'Filtered Matches: {len(matches_filtered)} matches')
print(f'Filtered Momentum: {len(momentum_filtered)} momentum records')
print(f'\nMatches in both datasets: {len(set(matches_filtered["id"]) & set(momentum_filtered["id"]))}')

In [ ]:
# Merge matches and momentum data on the 'id' field
# Use 'id' for the key and rename columns to avoid conflicts
merged_data = pd.merge(
    matches_filtered,
    momentum_filtered,
    on='id',
    suffixes=('_match', '_momentum'),
    how='inner'
)

print(f'Merged data shape: {merged_data.shape}')
print(f'\nMerged columns: {list(merged_data.columns)}')
print(f'\nFirst few rows:')
print(merged_data.head())

## Regression Discontinuity Design (RDD) Analysis for Hydration Breaks

In [ ]:
# Extract all hydration breaks from the dataset
hydration_breaks = []

for idx, row in merged_data.iterrows():
    match_id = row['id']
    stoppages = row['stoppages']
    
    # Filter for hydration breaks
    if isinstance(stoppages, list):
        for stoppage in stoppages:
            if len(stoppage) >= 2 and stoppage[1] == 'hydration':
                hydration_breaks.append({
                    'match_id': match_id,
                    'minute': stoppage[0],
                    'type': stoppage[1],
                    'duration': stoppage[2] if len(stoppage) > 2 else None
                })

hydration_df = pd.DataFrame(hydration_breaks)
print(f'Total hydration breaks identified: {len(hydration_df)}')
print(f'\nHydration breaks distribution:')
print(hydration_df.head(10))
print(f'\nMinutes of hydration breaks:')
print(hydration_df['minute'].describe())

In [ ]:
# Extract momentum series data for RDD analysis
rdd_data = []

for idx, row in merged_data.iterrows():
    match_id = row['id']
    series = row['series']  # [[time, momentum], ...]
    stoppages = row['stoppages']
    home_team = row['home_match']
    away_team = row['away_match']
    home_score = row['home_score']
    away_score = row['away_score']
    league = row['league']
    stage = row['stage']
    
    # Convert series to a more usable format
    if isinstance(series, list):
        momentum_series = pd.DataFrame(series, columns=['minute', 'momentum'])
    else:
        continue
    
    # Find hydration breaks for this match
    if isinstance(stoppages, list):
        for stoppage in stoppages:
            if len(stoppage) >= 2 and stoppage[1] == 'hydration':
                hydration_minute = stoppage[0]
                
                # Store all momentum data points for this match and hydration break
                rdd_data.append({
                    'match_id': match_id,
                    'home_team': home_team,
                    'away_team': away_team,
                    'home_score': home_score,
                    'away_score': away_score,
                    'league': league,
                    'stage': stage,
                    'hydration_minute': hydration_minute,
                    'momentum_series': momentum_series.copy()
                })

print(f'Matches with hydration breaks: {len(rdd_data)}')
print(f'\nExample momentum series from first match:')
print(rdd_data[0]['momentum_series'].head(10))

In [ ]:
# Extract data around hydration breaks for RDD analysis
def extract_rdd_window(momentum_series, treatment_minute, window_size=10):
    """
    Extract momentum data x minutes before and after treatment (hydration break).
    Returns data with running variable (distance from treatment).
    """
    # Filter data within the window
    lower_bound = treatment_minute - window_size
    upper_bound = treatment_minute + window_size
    
    windowed_data = momentum_series[
        (momentum_series['minute'] >= lower_bound) & 
        (momentum_series['minute'] <= upper_bound)
    ].copy()
    
    if len(windowed_data) == 0:
        return None
    
    # Create running variable: distance from treatment
    # Negative values = before treatment, positive = after
    windowed_data['running_variable'] = windowed_data['minute'] - treatment_minute
    windowed_data['treatment'] = (windowed_data['running_variable'] >= 0).astype(int)
    
    return windowed_data

# Extract all RDD windows
all_rdd_windows = []

for entry in rdd_data:
    window_df = extract_rdd_window(
        entry['momentum_series'],
        entry['hydration_minute'],
        window_size=10
    )
    
    if window_df is not None:
        window_df['match_id'] = entry['match_id']
        window_df['hydration_minute'] = entry['hydration_minute']
        window_df['home_team'] = entry['home_team']
        window_df['away_team'] = entry['away_team']
        window_df['home_score'] = entry['home_score']
        window_df['away_score'] = entry['away_score']
        window_df['league'] = entry['league']
        window_df['stage'] = entry['stage']
        all_rdd_windows.append(window_df)

rdd_combined = pd.concat(all_rdd_windows, ignore_index=True)

print(f'RDD windows extracted: {len(all_rdd_windows)}')
print(f'Total data points in RDD windows: {len(rdd_combined)}')
print(f'\nFirst few RDD data points:')
print(rdd_combined.head(10))
print(f'\nData summary:')
print(rdd_combined.groupby('treatment')[['momentum', 'running_variable']].agg(['mean', 'std', 'count']))

## RDD Visualization

In [ ]:
# Create RDD plot
fig, ax = plt.subplots(figsize=(12, 7))

# Scatter plot with treatment indicator
before = rdd_combined[rdd_combined['treatment'] == 0]
after = rdd_combined[rdd_combined['treatment'] == 1]

ax.scatter(before['running_variable'], before['momentum'], 
           alpha=0.5, label='Before Hydration Break', color='blue', s=30)
ax.scatter(after['running_variable'], after['momentum'], 
           alpha=0.5, label='After Hydration Break', color='red', s=30)

# Add vertical line at treatment point
ax.axvline(x=0, color='black', linestyle='--', linewidth=2, label='Hydration Break')

# Fit local regressions on each side
if len(before) > 1:
    z_before = np.polyfit(before['running_variable'], before['momentum'], 1)
    p_before = np.poly1d(z_before)
    x_before = np.linspace(before['running_variable'].min(), -0.1, 100)
    ax.plot(x_before, p_before(x_before), 'b-', linewidth=2, label='Trend Before')
    print(f'Before regression slope: {z_before[0]:.4f}')

if len(after) > 1:
    z_after = np.polyfit(after['running_variable'], after['momentum'], 1)
    p_after = np.poly1d(z_after)
    x_after = np.linspace(0.1, after['running_variable'].max(), 100)
    ax.plot(x_after, p_after(x_after), 'r-', linewidth=2, label='Trend After')
    print(f'After regression slope: {z_after[0]:.4f}')

ax.set_xlabel('Minutes from Hydration Break (negative = before, positive = after)', fontsize=11)
ax.set_ylabel('Momentum', fontsize=11)
ax.set_title('Regression Discontinuity Design: Effect of Hydration Breaks on Momentum', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## RDD Regression Analysis (with and without controls)

In [ ]:
# STEP 1: Prepare controls/covariates
# Controls help isolate the treatment effect by accounting for other factors

rdd_model = rdd_combined.copy()

# Encode categorical variables
le_home = LabelEncoder()
le_away = LabelEncoder()
le_league = LabelEncoder()

rdd_model['home_team_encoded'] = le_home.fit_transform(rdd_model['home_team'])
rdd_model['away_team_encoded'] = le_away.fit_transform(rdd_model['away_team'])
rdd_model['league_encoded'] = le_league.fit_transform(rdd_model['league'])

# Create additional control variables
rdd_model['goal_differential'] = rdd_model['home_score'] - rdd_model['away_score']
rdd_model['total_goals'] = rdd_model['home_score'] + rdd_model['away_score']
rdd_model['is_closing_stage'] = (rdd_model['stage'].astype(float) >= 3).astype(int)  # Knockout stages
rdd_model['abs_running_variable'] = np.abs(rdd_model['running_variable'])

print('Controls prepared:')
print(f'  - home_team_encoded: {rdd_model["home_team_encoded"].nunique()} unique teams')
print(f'  - away_team_encoded: {rdd_model["away_team_encoded"].nunique()} unique teams')
print(f'  - league_encoded: {rdd_model["league_encoded"].nunique()} unique leagues')
print(f'  - goal_differential: range [{rdd_model["goal_differential"].min()}, {rdd_model["goal_differential"].max()}]')
print(f'  - total_goals: range [{rdd_model["total_goals"].min()}, {rdd_model["total_goals"].max()}]')
print(f'  - is_closing_stage: {rdd_model["is_closing_stage"].value_counts().to_dict()}')
print(f'\nFirst few rows with controls:')
print(rdd_model[['momentum', 'treatment', 'running_variable', 'goal_differential', 'total_goals', 'is_closing_stage']].head())

In [ ]:
# MODEL 1: Simple RDD (no controls)
print('\n' + '='*70)
print('MODEL 1: SIMPLE RDD (No Controls)')
print('='*70)

X_simple = rdd_model[['running_variable', 'treatment']]
X_simple = sm.add_constant(X_simple)
y = rdd_model['momentum']

model_simple = sm.OLS(y, X_simple).fit()
print(model_simple.summary())

print('\nKey Finding (Model 1):')
print(f'Treatment Effect: {model_simple.params["treatment"]:.4f}')
print(f'P-value: {model_simple.pvalues["treatment"]:.4f}')

In [ ]:
# MODEL 2: RDD with controls
print('\n' + '='*70)
print('MODEL 2: RDD WITH CONTROLS')
print('='*70)

X_with_controls = rdd_model[[
    'running_variable', 
    'treatment',
    'goal_differential',      # Match score difference
    'total_goals',            # Total goals in match
    'is_closing_stage',       # Stage of tournament
    'league_encoded'          # League/competition
]]
X_with_controls = sm.add_constant(X_with_controls)

model_controls = sm.OLS(y, X_with_controls).fit()
print(model_controls.summary())

print('\nKey Finding (Model 2):')
print(f'Treatment Effect: {model_controls.params["treatment"]:.4f}')
print(f'P-value: {model_controls.pvalues["treatment"]:.4f}')

In [ ]:
# MODEL 3: RDD with extended controls (including interactions)
print('\n' + '='*70)
print('MODEL 3: RDD WITH EXTENDED CONTROLS (Interactions)')
print('='*70)

# Create interaction terms
rdd_model['treatment_x_running'] = rdd_model['treatment'] * rdd_model['running_variable']
rdd_model['treatment_x_goal_diff'] = rdd_model['treatment'] * rdd_model['goal_differential']

X_extended = rdd_model[[
    'running_variable', 
    'treatment',
    'treatment_x_running',    # Different slope before/after
    'goal_differential',
    'total_goals',
    'is_closing_stage',
    'league_encoded',
    'treatment_x_goal_diff'   # Effect modifier
]]
X_extended = sm.add_constant(X_extended)

model_extended = sm.OLS(y, X_extended).fit()
print(model_extended.summary())

print('\nKey Finding (Model 3):')
print(f'Treatment Effect: {model_extended.params["treatment"]:.4f}')
print(f'P-value: {model_extended.pvalues["treatment"]:.4f}')

In [ ]:
# COMPARE MODELS
print('\n' + '='*70)
print('MODEL COMPARISON')
print('='*70)

comparison = pd.DataFrame({
    'Model 1 (Simple)': {
        'Treatment Effect': model_simple.params['treatment'],
        'P-value': model_simple.pvalues['treatment'],
        'R-squared': model_simple.rsquared,
        'Adj. R-squared': model_simple.rsquared_adj
    },
    'Model 2 (With Controls)': {
        'Treatment Effect': model_controls.params['treatment'],
        'P-value': model_controls.pvalues['treatment'],
        'R-squared': model_controls.rsquared,
        'Adj. R-squared': model_controls.rsquared_adj
    },
    'Model 3 (Extended)': {
        'Treatment Effect': model_extended.params['treatment'],
        'P-value': model_extended.pvalues['treatment'],
        'R-squared': model_extended.rsquared,
        'Adj. R-squared': model_extended.rsquared_adj
    }
})

print(comparison.T)
print('\nInterpretation:')
print('- If treatment effect changes significantly across models, confounding may be present')
print('- Higher R-squared suggests better model fit')
print('- Controls help isolate the true causal effect of hydration breaks')

In [ ]:
# Additional statistics
from scipy.stats import ttest_ind

before_momentum = rdd_combined[rdd_combined['treatment'] == 0]['momentum']
after_momentum = rdd_combined[rdd_combined['treatment'] == 1]['momentum']

t_stat, p_value = ttest_ind(before_momentum, after_momentum)
mean_diff = after_momentum.mean() - before_momentum.mean()

print('\nT-Test Results (Before vs After):')
print('=' * 50)
print(f'Mean momentum before hydration: {before_momentum.mean():.4f} (std: {before_momentum.std():.4f})')
print(f'Mean momentum after hydration:  {after_momentum.mean():.4f} (std: {after_momentum.std():.4f})')
print(f'Mean difference: {mean_diff:.4f}')
print(f'T-statistic: {t_stat:.4f}')
print(f'P-value: {p_value:.4f}')
print('=' * 50)
if p_value < 0.05:
    print('Result: SIGNIFICANT difference (p < 0.05)')
else:
    print('Result: NO significant difference (p >= 0.05)')

## Inspect Data

In [ ]:
# Display first few rows of each dataframe
for name, df in parquet_data.items():
    print(f'\n{name}:')
    print(df.head())

In [ ]:
# Display merged data summary
print(f'Merged data info:')
print(f'Shape: {merged_data.shape}')
print(f'\nData types:')
print(merged_data.dtypes)
print(f'\nFirst 3 merged records:')
merged_data.head(3)